# AutoETF Local Research Agent

这个 Notebook 只面向本地测试。它读取本地 parquet 快照，不依赖 BigQuant 数据环境。

先确保 `data/parquet/local_etf_daily.parquet` 已经存在，再运行下面的聊天单元。

In [ ]:
import sys
from dataclasses import replace
from pathlib import Path

for p in [Path.cwd(), Path.cwd().parent]:
    if (p / 'src').exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        project_root = p
        break

from src.config import DEFAULT_CONFIG
from src.data_fetch_akshare import build_local_etf_parquet

local_parquet = project_root / 'data/parquet/local_etf_daily.parquet'
metadata_path = project_root / 'data/parquet/local_etf_metadata.json'

if not local_parquet.exists():
    print('local parquet missing, fetching from AKShare...')
    build_local_etf_parquet(
        symbols=['510300', '159915'],
        start_date='2024-01-01',
        end_date='2024-06-30',
        output_path=str(local_parquet),
        metadata_path=str(metadata_path),
        data_source='auto',
    )
    print('local parquet fetched and saved:', local_parquet)

config = replace(
    DEFAULT_CONFIG,
    data_backend='local',
    local_etf_parquet=str(local_parquet),
    local_benchmark_parquet=None,
)

print('project_root:', project_root)
print('local parquet exists:', local_parquet.exists())


project_root: /Users/yangpeng/Desktop/ai量化
local parquet exists: True


In [ ]:
import pandas as pd

path = project_root / "data/parquet/local_etf_daily.parquet"
df = pd.read_parquet(path)

print("shape:", df.shape)
print("columns:")
for i, c in enumerate(df.columns):
    print(i, c)

keywords = [
    "volume", "vol", "amount", "turn", "turnover",
    "ratio", "rate", "money", "close", "open", "high", "low"
]

matched_cols = [
    c for c in df.columns
    if any(k.lower() in c.lower() for k in keywords)
]

print("\nmatched_cols:")
matched_cols

# 换手率字段检查（注意 return_1d 等也会被 turn 命中，需单独筛）
turn_cols = [c for c in df.columns if "turnover" in c.lower() or c.lower() in {"turn", "turnover_rate"}]
print("turnover cols:", turn_cols)

# 拉 159638 最近一天，方便外部核查
code_col = "instrument" if "instrument" in df.columns else "code"
show_cols = [
    "date", code_col, "name", "open", "high", "low", "close",
    "volume", "amount",
    "volume_ratio_5d", "volume_ratio_20d",
    "amount_ratio_5d", "amount_ratio_20d",
    "momentum_20d", "trend_strength", "volatility_20d",
]

sub = df[df[code_col].astype(str) == "159638"].copy()
sub["date"] = pd.to_datetime(sub["date"])
latest = sub["date"].max()
row = sub.loc[sub["date"] == latest, show_cols]

print(f"\n159638 最新交易日: {latest.date()}")
display(row)

print("\n最近 5 个交易日:")
display(sub[show_cols].tail(5))

In [ ]:
import getpass
from IPython.display import clear_output

from src.notebook_reload import reload_autoetf_runtime
from src.notebook_chat_ui import launch_notebook_chat
from src.secret_loader import load_deepseek_api_key

reload_autoetf_runtime()

api_key = load_deepseek_api_key(project_root)
if not api_key:
    api_key = getpass.getpass('DeepSeek API Key:')

clear_output(wait=True)

chat_ui = launch_notebook_chat(
    config=config,
    use_llm=True,
    api_key=api_key,
    model='deepseek-v4-flash',
)
print('AutoETF chat UI launched. Re-run this cell to refresh code; duplicate replies should no longer appear.')


AutoETF chat UI launched. If you do not see widgets, restart the kernel and run this cell again.
